In [ ]:
!nvidia-smi

In [3]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Explicitly use CPU for local stability
device = torch.device("cpu")
print(f"Running on: {device}")

Running on: cpu


## 1. Data Loading & Filtering

In [4]:
# Load Labels
csv_path = 'labels_synthetic_calibrated_with_path.csv'

# Check if file exists (Colab path might be different, e.g., /content/labels...)
if not os.path.exists(csv_path):
    print(f"Warning: {csv_path} not found. Please update 'csv_path' to your location.")
    # Example for Colab upload:
    # csv_path = '/content/labels_synthetic_calibrated.csv'

df = pd.read_csv(csv_path)

# Filter for Mobile images
# The 'source' column should distinguish between 'mobile' and 'slit_lamp'
print("Available sources:", df['source'].unique())

target_source = 'mobile'
if target_source not in df['source'].values:
    print("Error: 'mobile' source not found in CSV.")

mobile_df = df[df["source"] == target_source].copy()
print(f"Filtered {len(mobile_df)} mobile images.")

# Ensure relative path column exists
if 'relative_path' not in mobile_df.columns:
    # Fallback: construct path if needed, or error out
    print("Error: 'relative_path' column missing. Please ensure CSV is updated.")
else:
    print("Path column found.")

Available sources: <StringArray>
['mobile', 'slit_lamp']
Length: 2, dtype: str
Filtered 13338 mobile images.
Path column found.


## 2. Dataset Definition

In [5]:
class CataractDataset(Dataset):
    def __init__(self, dataframe, root_dir=None, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir if root_dir else ''
        self.transform = transform

        # Targets: NO, NC, CO, PSC
        self.targets = ['NO_pseudo', 'NC_pseudo', 'CO_pseudo', 'PSC_pseudo']

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        # Construct image path
        # If running in Colab, root_dir might be '/content/'
        img_rel_path = row['relative_path'].replace('\\', '/')
        img_path = os.path.join(self.root_dir, img_rel_path)

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a dummy tensor or handle error (here we fail noisy)
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        # Get labels (continuous or discrete? The concept model predicts continuous 0-5)
        # If using pseudo-labels (bins), we can convert to float
        labels = row[self.targets].values.astype(float)
        labels = torch.tensor(labels, dtype=torch.float32)

        return image, labels

## 3. Train/Val Split & DataLoader

In [6]:
# Split into Train and Validation
train_df, val_df = train_test_split(mobile_df, test_size=0.2, random_state=42)

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Datasets
# Use current working directory as root (where the CSV paths are relative to)
data_root = '.'
train_dataset = CataractDataset(train_df, root_dir=data_root, transform=train_transform)
val_dataset = CataractDataset(val_df, root_dir=data_root, transform=val_transform)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

Train size: 10670, Val size: 2668


## 4. Model Definition

In [7]:
class ConceptPredictor(nn.Module):
    def __init__(self, backbone_name='resnet18', pretrained=True):
        super(ConceptPredictor, self).__init__()

        # Load backbone
        if backbone_name == 'resnet18':
            self.backbone = models.resnet18(pretrained=pretrained)
            in_features = self.backbone.fc.in_features
            # Remove the classification head
            self.backbone.fc = nn.Identity()
        elif backbone_name == 'efficientnet_b0':
            self.backbone = models.efficientnet_b0(pretrained=pretrained)
            in_features = self.backbone.classifier[1].in_features
            # Remove classification head
            self.backbone.classifier = nn.Identity()
        else:
            raise ValueError(f"Backbone {backbone_name} not supported.")

        # Concept Head: 4 neurons for NO, NC, CO, PSC
        self.concept_head = nn.Linear(in_features, 4)

    def forward(self, x):
        features = self.backbone(x)
        raw_output = self.concept_head(features)

        # Clamp output to valid range [0, 5]
        clamped_output = torch.clamp(raw_output, 0, 5)

        return clamped_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = ConceptPredictor(backbone_name='resnet18', pretrained=True).to(device)

Using device: cpu


c:\Python314\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Python314\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## 5. Training Loop

In [8]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10
best_val_loss = float('inf')

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

    val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}] Train Loss: {epoch_loss:.4f} Val Loss: {val_loss:.4f}")

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'concept_mobile.pt')
        print("  Model saved!")

print("Training Complete.")

Epoch [1/10] Train Loss: 0.6538 Val Loss: 0.1556
  Model saved!
Epoch [2/10] Train Loss: 0.1456 Val Loss: 0.1284
  Model saved!
Epoch [3/10] Train Loss: 0.1249 Val Loss: 0.1259
  Model saved!
Epoch [4/10] Train Loss: 0.1122 Val Loss: 0.1110
  Model saved!
Epoch [5/10] Train Loss: 0.1046 Val Loss: 0.1111
Epoch [6/10] Train Loss: 0.1003 Val Loss: 0.1107
  Model saved!
Epoch [7/10] Train Loss: 0.0937 Val Loss: 0.1030
  Model saved!
Epoch [8/10] Train Loss: 0.0886 Val Loss: 0.0941
  Model saved!
Epoch [9/10] Train Loss: 0.0844 Val Loss: 0.0998
Epoch [10/10] Train Loss: 0.0797 Val Loss: 0.1011
Training Complete.


In [9]:
save_dir = 'mobile_train'
model_filename = 'concept_mobile_final.hdf5'
save_path = os.path.join(save_dir, model_filename)

os.makedirs(save_dir, exist_ok=True)

# Save the model's state dictionary
torch.save(model.state_dict(), save_path)

print(f"Model saved to: {save_path}")

Model saved to: mobile_train\concept_mobile_final.hdf5
